In [ ]:
import numpy as np
import os
import sys
import matplotlib.pyplot as plt
from pathlib import Path
from benchmarking_sim.quadrotor.benchmark_util.utils import plot_colors

In [ ]:
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
data_folder = 'gpmpc_acados_TP/results'
data_folder_path = os.path.join(notebook_dir, data_folder)
assert os.path.exists(data_folder_path), 'data_folder_path does not exist'

s = 2 # times of std

In [ ]:
def extract_rollouts(notebook_dir, data_folder, controller_name, additional=''):
    data_folder_path = os.path.join(notebook_dir, controller_name, data_folder)
    assert os.path.exists(data_folder_path), 'data_folder_path does not exist'

    # find all the subfolders in the data_folder_path
    subfolders = [f.path for f in os.scandir(data_folder_path) if f.is_dir()]
    # load the row 'rmse in the metrics.txt
    metrics = []
    traj_resutls = []
    for subfolder in subfolders:
        file_path = os.path.join(subfolder, 'metrics.txt')
        with open(file_path, 'r') as file:
            lines = file.readlines()
            for line in lines:
                if not line.startswith('rmse_std') and line.startswith('rmse'):
                    # split the text between : and \n
                    line = line.split(': ')[-1].split('\n')[0]
                    metrics.append(eval(line))

        # find the file ends with pickle and get the data
        for file in os.listdir(subfolder):
            if file.endswith('.pkl'):
                file_path = os.path.join(subfolder, file)
                results = np.load(file_path, allow_pickle=True)
                traj_data = results['trajs_data']['obs'][0]
                traj_resutls.append(traj_data)

    traj_resutls = np.array(traj_resutls)
    traj_file_name = f'traj_results_{controller_name}{additional}.npy'
    np.save(traj_file_name, traj_resutls)
    rmse_mean_mpc = np.mean(metrics)
    rmse_std_mpc = np.std(metrics)
    print(f'rmse_{controller_name}{additional}', rmse_mean_mpc, rmse_std_mpc)
    return traj_resutls, metrics


In [ ]:


class benchmark_rmse_data:
    # data stored in the path like "prior/seed/figs/"
    # each data is a csv file
    def __init__(self, data_folder_path, controller_name, prior_name, dt):
        self.data_folder_path = Path(data_folder_path)
        self.controller_name = controller_name
        self.prior_name = prior_name
        self.controller_data_folder_path = self.data_folder_path / controller_name
        self.prior_data_folder_path = self.controller_data_folder_path / prior_name
        self.none_count = 0
        self.early_stop = 0
        self.max_seed = None
        self.check_data_folder()
        self.find_all_seed_folders()
        self.append_figs_to_seeds()
        self.load_csv_cost_data_all_seeds()
        self.early_stop_ratio = self.early_stop / self.max_seed
        self.dt = dt

    def check_data_folder(self):
        if not self.prior_data_folder_path.exists():
            print('prior data folder does not exist')
            return False
        print(f'prior data {self.prior_name} folder exists')
        return True
    
    def find_all_seed_folders(self):
        # find all folder names in the prior data folder
        self.seed_folders = [f for f in self.prior_data_folder_path.iterdir() if f.is_dir()]
        # Extract seed numbers and sort folders
        seed_list = [int(f.name.split('seed')[1].split('_')[0]) for f in self.seed_folders]
        self.seed_folders = sorted(self.seed_folders, key=lambda x: int(x.name.split('seed')[1].split('_')[0]))
        self.max_seed = max(seed_list)
        print('max seed', self.max_seed)
        ''' uncomment the following line for fewer seeds '''
    
    def append_figs_to_seeds(self):
        # Append 'figs' to the end of seed_folders
        self.seed_folders = [f / 'figs' for f in self.seed_folders]
    
    def load_csv_cost_data(self, seed):
        # Load the CSV file in the seed folder
        seed_folder = self.seed_folders[seed - 1]
        csv_file = seed_folder / 'rmse_error_learning_curve.csv'
        # If the file does not exist, return None
        if not csv_file.exists():
            print(f'csv file for seed {seed} does not exist')
            self.none_count += 1
            return None
        data = np.genfromtxt(csv_file, delimiter=',')
        return data

    def load_csv_cost_data_all_seeds(self):
        # Load all CSV files in the seed folders
        self.data_all_seeds = []
        for seed in range(len(self.seed_folders)):
            data = self.load_csv_cost_data(seed)
            self.data_all_seeds.append(data)
        all_epoch = [0 for _ in range(len(self.seed_folders))]
        for data in self.data_all_seeds:
            if data is not None:
                all_epoch.append(data.shape[0])
        self.sim_epoch = max(all_epoch)
        print('sim_epoch', self.sim_epoch)
        for i, data in enumerate(self.data_all_seeds):
            if data is not None:
                if data.shape[0] < self.sim_epoch:
                    self.early_stop += 1
                    print(f'seed {i} early stop with epoch {data.shape[0]}')
            elif data is None:
                self.early_stop += 1
                print(f'seed {i} early stop with epoch 0')
        # Filter out data with None and early stop
        self.merged_data = [data for data in self.data_all_seeds if data is not None and data.shape[0] == self.sim_epoch]
        
    def get_mean_std(self):
        # Get the mean and std of the data
        self.mean_data = np.mean(self.merged_data, axis=0)
        self.std_data = np.std(self.merged_data, axis=0)
        # Round the first column of mean to integer
        self.mean_data[:, 0] = np.round(self.mean_data[:, 0])
        # Exclude the first column of the std
        self.std_data = self.std_data[:, 1:].squeeze()
        # Modify the data index axis with dt
        self.mean_data[:, 0] = self.mean_data[:, 0] * self.dt
        return self.mean_data, self.std_data

In [ ]:
controller_name = ''
dt = 1/60
tag = 'hpo'
# tag = 'handtuned'
id_type = ''
# id_type = '_dw_h=1dot5'
# id_type = '_dw_h=2'
# id_type = '_dw_h=2dot5'
# id_type = '_dw_h=3'
# id_type = '_dw_h=4'
# id_type = '_ob_ns=5'
# id_type = '_ob_ns=10'
# id_type = '_ob_ns=15'
# id_type = '_ob_ns=20'
# id_type = '_proc_ns=3'
# id_type = '_proc_ns=5'
# id_type = '_proc_ns=7'
# id_type = '_proc_ns=10'
# id_type = '_ob_ns=5_proc_ns=3'
# id_type = '_ob_ns=10_proc_ns=5'
# id_type = '_ob_ns=15_proc_ns=7'
# id_type = '_ob_ns=20_proc_ns=10'
# id_type = '_param'
# id_type = '_tr'

prior_hpo = f'{tag}{id_type}/temp'
data_hpo = benchmark_rmse_data(data_folder_path, controller_name, prior_hpo, dt)
mean_hpo, std_hpo = data_hpo.get_mean_std()
mean_hpo[0, 0] = 1

# get the 25 % and 75 % quantile
merged_data = np.array(data_hpo.merged_data)
rmse_data = merged_data[:, :, 1]

In [ ]:
print('rmse_data.shape', rmse_data.shape)
rmse_mean = np.mean(rmse_data, axis=0)
rmse_std = np.std(rmse_data, axis=0)

train_steps_seconds = mean_hpo[:, 0]
train_steps_data = np.arange(0, len(train_steps_seconds)) * 20
train_steps_seconds = train_steps_data / 20 * 11
train_steps = (train_steps_seconds / dt).astype(int)

train_steps_data[0] = 1
train_steps[0] = 1
train_steps_seconds[0] = 1

In [ ]:
saved_results = {
    'rmse': rmse_data,
    'rmse_mean': rmse_mean,
    'rmse_std': rmse_std,
    'train_steps': train_steps,
    'train_steps_data': train_steps_data,
    'train_steps_seconds': train_steps_seconds
}
np.save(f'./data/gpmpc_acados_TP_{tag}{id_type}_convergence_results.npy', saved_results)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 2))
ax.plot(train_steps_seconds, rmse_mean, label='GP-MPC', color=plot_colors['GP-MPC'])
ax.fill_between(train_steps_seconds, rmse_mean - s * rmse_std, rmse_mean + s * rmse_std, alpha=0.2, color=plot_colors['GP-MPC'])
# print the number next to the point
for i, txt in enumerate(rmse_mean):
    ax.annotate(f'{txt:.4f}', (train_steps_seconds[i], rmse_mean[i]), textcoords="offset points", xytext=(0,10), ha='center')
ax.legend(ncol=2, loc='upper right')
ax.set_ylim([0, None])
ax.set_xlabel('Train steps [s]')
ax.set_ylabel('RMSE')
ax.set_title(f'GPMPC convergence_{tag}{id_type}')

save_name = f'GPMPC_convergence_{tag}{id_type}'
if not os.path.exists(f'{notebook_dir}/plotting/convergence'):
    os.makedirs(f'{notebook_dir}/plotting/convergence')
fig.savefig(f'{notebook_dir}/plotting/convergence/{save_name}.png', bbox_inches='tight')